# 02 - Lakebase: operational serving for the app

1. Create a Lakebase (Postgres) project.
2. Register its database in Unity Catalog, so Postgres tables (including the
   approvals the app writes) are governed and queryable from the lakehouse.
3. Sync the gold tables from Delta into Postgres (synced tables) for the app to read.

In [1]:
import logging, time, warnings, yaml
warnings.filterwarnings("ignore")
logging.getLogger("databricks.sdk").setLevel(logging.ERROR)
from databricks.connect import DatabricksSession
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service import postgres as pg

cfg = yaml.safe_load(open("../config.yml"))
spark = DatabricksSession.builder.profile(cfg["profile"]).serverless(True).getOrCreate()
w = WorkspaceClient(profile=cfg["profile"])
cat, sch, project, lb_cat = cfg["catalog"], cfg["schema"], cfg["lakebase_project"], cfg["lakebase_catalog"]
branch = f"projects/{project}/branches/production"

# Gold tables served to the app, with their primary keys.
SYNCED = {"gold_desk_stress": ["scenario_id", "desk"], "gold_commentary": ["scenario_id"]}

In [2]:
try:
    w.postgres.get_project(f"projects/{project}")
except NotFound:
    w.postgres.create_project(project_id=project, project=pg.Project(
        spec=pg.ProjectSpec(display_name="Stress testing demo"))).wait()
try:
    w.postgres.get_catalog(f"catalogs/{lb_cat}")
except NotFound:
    w.postgres.create_catalog(catalog_id=lb_cat, catalog=pg.Catalog(spec=pg.CatalogCatalogSpec(
        postgres_database="databricks_postgres", branch=branch))).wait()
print("project:", project, "| UC catalog:", lb_cat, "->", w.postgres.get_catalog(f"catalogs/{lb_cat}").status.branch)

project: stress-demo | UC catalog: stress_demo_lakebase -> projects/stress-demo/branches/production


## Synced tables (Delta -> Postgres)
Snapshot mode: notebook 01 rebuilds gold on every run, so each run re-syncs the full table.

In [3]:
for table, keys in SYNCED.items():
    name = f"{lb_cat}.{sch}.{table}"
    try:
        st = w.postgres.get_synced_table(f"synced_tables/{name}")
        w.pipelines.start_update(st.status.pipeline_id)  # refresh the existing sync
    except NotFound:
        w.postgres.create_synced_table(synced_table_id=name, synced_table=pg.SyncedTable(
            spec=pg.SyncedTableSyncedTableSpec(
                source_table_full_name=f"{cat}.{sch}.{table}",
                primary_key_columns=keys,
                scheduling_policy=pg.SyncedTableSyncedTableSpecSyncedTableSchedulingPolicy.SNAPSHOT,
                branch=branch, postgres_database="databricks_postgres",
                create_database_objects_if_missing=True,
                new_pipeline_spec=pg.NewPipelineSpec(storage_catalog=cat, storage_schema=sch)))).wait()

time.sleep(30)  # let the refresh start before polling
for table in SYNCED:
    while True:
        state = w.postgres.get_synced_table(f"synced_tables/{lb_cat}.{sch}.{table}").status.detailed_state.value
        if "PROVISIONING" not in state and "UPDATE" not in state.replace("NO_PENDING_UPDATE", ""):
            break
        time.sleep(20)
    print(f"{table}: {state}")

gold_desk_stress: SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE


gold_commentary: SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE


## Read the served data back through the Lakebase catalog in Unity Catalog

In [4]:
spark.sql(f"""
  SELECT scenario_name, desk, round(pnl_jpy / 1e9, 1) AS pnl_jpy_bn, limit_utilization, limit_breach
  FROM {lb_cat}.{sch}.gold_desk_stress ORDER BY scenario_id, desk
""").show(truncate=False)
spark.sql(f"SELECT scenario_id, left(commentary, 80) AS commentary FROM {lb_cat}.{sch}.gold_commentary ORDER BY 1").show(truncate=False)

+----------------------+------+----------+-----------------+------------+
|scenario_name         |desk  |pnl_jpy_bn|limit_utilization|limit_breach|
+----------------------+------+----------+-----------------+------------+
|BoJ hikes 100bp       |Credit|-0.3      |0.071            |false       |
|BoJ hikes 100bp       |Equity|-1.1      |0.216            |false       |
|BoJ hikes 100bp       |FX    |-3.7      |0.459            |false       |
|BoJ hikes 100bp       |Rates |-0.3      |0.147            |false       |
|Yen surges 15%        |Credit|-0.5      |0.134            |false       |
|Yen surges 15%        |Equity|-1.6      |0.324            |false       |
|Yen surges 15%        |FX    |-11.6     |1.455            |true        |
|Yen surges 15%        |Rates |0.1       |-0.054           |false       |
|Global equity sell-off|Credit|-3.0      |0.747            |false       |
|Global equity sell-off|Equity|-5.0      |0.994            |false       |
|Global equity sell-off|FX    |-6.7   

+-----------+--------------------------------------------------------------------------------+
|scenario_id|commentary                                                                      |
+-----------+--------------------------------------------------------------------------------+
|S1         |The FX desk drives the stressed loss, with JPY -3.7bn of the JPY -5.3bn total un|
|S2         |The FX desk drives the stressed loss, with USDJPY (JPY -7.2bn) and EURJPY (JPY -|
|S3         |The FX desk drives the largest stressed loss, JPY -6.7bn of the JPY -14.3bn tota|
|S4         |The stressed loss of JPY -9.1bn in the credit-spread-widening scenario is led by|
+-----------+--------------------------------------------------------------------------------+

